<a href="https://colab.research.google.com/github/tinaba96/ai_clustering/blob/main/kmeans_bottomup_outlier_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### コンストラクティブフィードバック 分析V2 k-means bottom-up

### バージョン内容
- ベクトル化時に外れ値を特定・除外してクラスタリングを行う
- 除外されたvoeは「その他」としてグルーピング

### 使い方
1. パラメータを設定してください
2. (1回目)「すべてのセルを実行」
3. (2回目以降)
   - targetを変えたら、データの読み込み以降を実行
   - 外れ値割合（contamination)を変えたら、ベクトル化以降を実施
   - クラスタリング数を変えたら、クラスタリング以降を実行

In [1]:
configs = {
    'sample-1': 'challenge_voe.ndjson',
    'sample-2': 'strength_voe.ndjson',
    '1-1': '',
    '1-2': '',
    '1-3': '',
    '1-4': '',
    '2-1': '',
    '2-1': '',
    '2-1': '',
    '2-1': '',
    '3-1': '',
    '3-1': '',
    '3-1': '',
    '3-1': '',
    '4-1': 'analysis-result_tenant_id=0198ea05-0960-7add-87b7-d2cbd9646bb9_survey_id=0199280e-2067-7259-9f3c-41a7e064dd45_topic_id=0199a8ab-2df6-7bf9-b70e-2800ea3612ff_challenge_voe.ndjson',
    '4-2': 'analysis-result_tenant_id=0198ea05-0960-7add-87b7-d2cbd9646bb9_survey_id=0199280e-2067-7259-9f3c-41a7e064dd45_topic_id=0199a8ab-2df7-7c56-8171-06122f0cedb7_strength_voe.ndjson',
    '4-3': 'analysis-result_tenant_id=0198ea05-0960-7add-87b7-d2cbd9646bb9_survey_id=0199280e-2067-7259-9f3c-41a7e064dd45_topic_id=0199a8ab-2df7-7a9b-9154-ad768a881c36_challenge_voe.ndjson',
    '4-4': 'analysis-result_tenant_id=0198ea05-0960-7add-87b7-d2cbd9646bb9_survey_id=0199280e-2067-7259-9f3c-41a7e064dd45_topic_id=0199a8ab-2df7-7fef-9412-b44eb7b9b9e1_challenge_voe.ndjson',
}
target = '4-4' #@param ['sample-1', 'sample-2', '1-1', '1-2', '1-3', '1-4', '2-1', '2-2', '2-3', '2-4', '3-1', '3-2', '3-3', '3-4', '4-1', '4-2', '4-3', '4-4']
num_root_clusters = 8 #@param {type:"integer"}
num_leaf_clusters = 35 #@param {type:"integer"}
use_clustering_summary = False #@param {type:"boolean"}

data_dir = '/content/drive/Shareddrives/00.U-ZERO/03.development/02.コンストラクティブフィードバック/分析V2 Colab/data/'
input_file = f"{data_dir}/{configs.get(target)}"

print(target)
print(input_file)
print(f"親クラスター数: {num_root_clusters}")
print(f"子クラスター数: {num_leaf_clusters}")


4-4
/content/drive/Shareddrives/00.U-ZERO/03.development/02.コンストラクティブフィードバック/分析V2 Colab/data//analysis-result_tenant_id=0198ea05-0960-7add-87b7-d2cbd9646bb9_survey_id=0199280e-2067-7259-9f3c-41a7e064dd45_topic_id=0199a8ab-2df7-7fef-9412-b44eb7b9b9e1_challenge_voe.ndjson
親クラスター数: 8
子クラスター数: 35


In [2]:
%pip install sentence-transformers scikit-learn pandas numpy umap-learn matplotlib

In [3]:
import json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import scipy.cluster.hierarchy as sch
import warnings
warnings.filterwarnings('ignore')

Googleドライブとの接続を許可してください

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### データの読み込み
**targetを変えたら、これより下を実行**

In [7]:
voes = []

with open(input_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    if lines:
        line = lines[-1]
        data = json.loads(line)
        for post_data in data.get('post_data_list', []):
            challenge_id = post_data.get('challenge_id')
            if 'challenge_id' in post_data:
              for challenge_voe in post_data.get('challenge_voes', []):
                  voes.append({
                      'voe_v2_id': challenge_voe.get('challenge_voe_v2_id'),
                      'summary': challenge_voe.get('challenge_summary'),
                      'clustering_summary': challenge_voe.get('clustering_challenge_summary' if use_clustering_summary else 'challenge_summary'),
                  })
            elif 'strength_id' in post_data:
              for strength_voe in post_data.get('strength_voes', []):
                  voes.append({
                      'voe_v2_id': strength_voe.get('strength_voe_v2_id'),
                      'summary': strength_voe.get('strength_summary'),
                      'clustering_summary': strength_voe.get('clustering_strength_summary' if use_clustering_summary else 'strength_summary'),
                  })

data_count = len(voes)
print(f"データ数: {data_count}")

データ数: 1104


# ベクトル化

**contaminationを0〜1の間で設定してください**

contamination: 外れ値割合（例: 0.05 = 全体の5%を外れ値として扱う）


In [35]:
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import IsolationForest

contamination = 0.1  # @param {type:"number"}
n_total = len(voes)

# 外れ値の期待件数
expected_outliers = int(n_total * contamination)
print(f"データ件数: {n_total} 件")
print(f"外れ値として扱う件数（予定）: {expected_outliers} 件 ({contamination*100:.1f}%)")


MODEL_NAME = 'intfloat/multilingual-e5-base'

model = SentenceTransformer(MODEL_NAME)

inputs = [voe['clustering_summary'] for voe in voes]
sentences = [f"passage: {input}" for input in inputs]

embeddings = model.encode(
    sentences,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"エンベディング形状: {embeddings.shape}")


# ------------------------------
# 外れ値検出
# ------------------------------
iso = IsolationForest(
    contamination = contamination,
    random_state=42
)
outlier_flags = iso.fit_predict(embeddings)  # -1: 外れ値, 1: 正常値

# ------------------------------
# 分離
# ------------------------------
normal_indices = np.where(outlier_flags == 1)[0]
outlier_indices = np.where(outlier_flags == -1)[0]

# 正常データ
normal_voes = [voes[i] for i in normal_indices]
normal_embeddings = embeddings[normal_indices]

# 外れ値データ
outlier_voes = [voes[i] for i in outlier_indices]
outlier_embeddings = embeddings[outlier_indices]

print(f"正常データ: {len(normal_voes)} 件")
print(f"外れ値: {len(outlier_voes)} 件")



データ件数: 1104 件
外れ値として扱う件数（予定）: 110 件 (10.0%)


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def evaluate_cluster_numbers(features, min_k, max_k):
    """
    複数の手法でクラスタ数を評価

    使用手法:
    1. シルエット法 (Silhouette Score) - 高いほど良い
    2. カリンスキー・ハラバス指数 (Calinski-Harabasz Index) - 高いほど良い
    3. デイビス・ボールディン指数 (Davies-Bouldin Index) - 低いほど良い
    4. エルボー法 (Elbow Method) - 慣性の変化率
    """
    if len(features) < min_k:
        print(f"データ数({len(features)})が最小クラスタ数({min_k})未満です")
        return

    k_range = range(min_k, max_k + 1)
    results = []

    print(f"クラスタ数評価開始 - 範囲: {min_k}-{max_k}")

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
        cluster_labels = kmeans.fit_predict(features)

        n_unique_labels = len(np.unique(cluster_labels))

        if n_unique_labels < 2:
            print(f"k={k}: 実際のクラスタ数が{n_unique_labels}のため評価指標をスキップ")
            continue

        silhouette_avg = silhouette_score(features, cluster_labels)
        calinski_score = calinski_harabasz_score(features, cluster_labels)
        davies_score = davies_bouldin_score(features, cluster_labels)
        inertia = kmeans.inertia_

        results.append({
            'k': k,
            'silhouette': silhouette_avg,
            'calinski': calinski_score,
            'davies': davies_score,
            'inertia': inertia
        })

        print(f"k={k}: S={silhouette_avg:.3f}, C={calinski_score:.1f}, D={davies_score:.3f}, I={inertia:.1f}")

    if results:
        best_silhouette = max(results, key=lambda x: x['silhouette'])
        best_calinski = max(results, key=lambda x: x['calinski'])
        best_davies = min(results, key=lambda x: x['davies'])

        print("\n=== 評価結果サマリー ===")
        print(f"シルエット法による推奨: k={best_silhouette['k']} (スコア: {best_silhouette['silhouette']:.3f})")
        print(f"カリンスキー・ハラバス法による推奨: k={best_calinski['k']} (スコア: {best_calinski['calinski']:.1f})")
        print(f"デイビス・ボールディン法による推奨: k={best_davies['k']} (スコア: {best_davies['davies']:.3f})")

    return results

def plot_elbow_method(results):
    k_values = [r['k'] for r in results]
    inertias = [r['inertia'] for r in results]

    plt.figure(figsize=(10, 6))
    plt.plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Number of Clusters (k)', fontsize=12)
    plt.ylabel('Inertia', fontsize=12)
    plt.title('Elbow Method', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

'''
suggested_min = max(2, data_count // 16)
suggested_max = max(2, data_count // 8)

print(f"評価範囲: {suggested_min} ~ {suggested_max}")
results = evaluate_cluster_numbers(embeddings, suggested_min, suggested_max)
plot_elbow_method(results)
'''

'\nsuggested_min = max(2, data_count // 16)\nsuggested_max = max(2, data_count // 8)\n\nprint(f"評価範囲: {suggested_min} ~ {suggested_max}")\nresults = evaluate_cluster_numbers(embeddings, suggested_min, suggested_max)\nplot_elbow_method(results)\n'

### クラスタリング実行

クラスタリング数を変えたら、これより下を実行

In [16]:
kmeans_leaf = KMeans(n_clusters=num_leaf_clusters, random_state=42, n_init=10, max_iter=300)
kmeans_leaf.fit(normal_embeddings)
leaf_labels = kmeans_leaf.labels_

print(f"子クラスター数: {len(kmeans_leaf.cluster_centers_)}")

子クラスター数: 35


In [18]:
def merge_clusters_with_hierarchy(
        cluster_centers: np.ndarray,
        kmeans_labels: np.ndarray,
        data_count: int,
        n_cluster_cut: int,
):
    """
    階層的クラスタリングを使用してクラスタをマージする

    Args:
        cluster_centers: K-Meansのクラスタ中心
        kmeans_labels: K-Meansによるクラスタラベル
        umap_array: 元データの配列
        n_cluster_cut: 最終的なクラスタ数

    Returns:
        final_labels: マージ後のクラスタラベル
    """
    # クラスタ中心に対して階層的クラスタリングを実行
    Z = sch.linkage(cluster_centers, method="ward")

    # 指定されたクラスタ数でカット
    cluster_labels_merged = sch.fcluster(Z, t=n_cluster_cut, criterion="maxclust")

    # 各データポイントの最終ラベルを決定
    final_labels = np.zeros(data_count, dtype=int)

    for i in range(data_count):
        original_label = kmeans_labels[i]
        # cluster_labels_mergedは1ベースなので、0ベースに変換
        final_labels[i] = cluster_labels_merged[original_label] - 1

    return final_labels

root_labels =  merge_clusters_with_hierarchy(
    cluster_centers=kmeans_leaf.cluster_centers_,
    kmeans_labels=kmeans_leaf.labels_,
    data_count=len(normal_voes),
    n_cluster_cut=num_root_clusters,
)

print(f"親クラスター数: {len(set(root_labels))}")

親クラスター数: 8


In [25]:
leaf_centers = kmeans_leaf.cluster_centers_

root_centers = {}
for root_id in sorted(set(root_labels)):
    mask = root_labels == root_id
    root_centers[root_id] = normal_embeddings[mask].mean(axis=0)

leaf_distances = []
root_distances = []

for i in range(len(voes)):
    if i in outlier_indices:
        # 外れ値は距離を NaN に
        leaf_distances.append(np.nan)
        root_distances.append(np.nan)
    else:
        # normal_indices 内での位置
        idx = np.where(normal_indices == i)[0][0]

        leaf_id = leaf_labels[idx]
        root_id_ = root_labels[idx]

        leaf_center = leaf_centers[leaf_id]
        root_center = root_centers[root_id_]

        leaf_distances.append(np.linalg.norm(normal_embeddings[idx] - leaf_center))
        root_distances.append(np.linalg.norm(normal_embeddings[idx] - root_center))


# 全データ用の root/leaf ラベル配列を作成
root_labels_full = np.array(["その他"] * len(voes), dtype=object)
leaf_labels_full = np.array([np.nan] * len(voes), dtype=object)  # 外れ値は NaN など

# 正常データのラベルを埋める
root_labels_full[normal_indices] = root_labels
leaf_labels_full[normal_indices] = leaf_labels

df_results = pd.DataFrame({
    'voe_v2_id': [voe['voe_v2_id'] for voe in voes],
    'summary': [voe['summary'] for voe in voes],
    'clustering_summary': [voe['clustering_summary'] for voe in voes],
    'root_cluster': root_labels_full,
    'leaf_cluster': leaf_labels_full,
    'root_distance': root_distances,
    'leaf_distance': leaf_distances
})

print(f"Root クラスタ数: {len(df_results['root_cluster'].unique())}, Leaf クラスタ数: {len(df_results['leaf_cluster'].unique())}")

# --- 外れ値の root_cluster を "その他" に置き換え ---
df_results.loc[outlier_indices, 'root_cluster'] = "その他"

# 確認
print(f"Root クラスタ数（外れ値含む）: {df_results['root_cluster'].nunique()}")
print(df_results['root_cluster'].value_counts())

root_ids = [rid for rid in df_results['root_cluster'].unique() if rid != "その他"]
root_ids = sorted(root_ids) + ["その他"]


for root_id in root_ids:
    root_df = df_results[df_results['root_cluster'] == root_id]
    print(f"Root {root_id}: {len(root_df)} 件")

Root クラスタ数: 9, Leaf クラスタ数: 36
Root クラスタ数（外れ値含む）: 9
root_cluster
6      457
5      264
3      146
4       95
その他     56
0       56
2       11
1       10
7        9
Name: count, dtype: int64
Root 0: 56 件
Root 1: 10 件
Root 2: 11 件
Root 3: 146 件
Root 4: 95 件
Root 5: 264 件
Root 6: 457 件
Root 7: 9 件
Root その他: 56 件


In [27]:
df_results_csv = df_results.copy()
df_results_csv['summary'] = df_results_csv['summary'].str.replace('\n', '\\n', regex=False)
df_results_csv['summary'] = df_results_csv['summary'].str.replace('\r', '\\r', regex=False)

df_results_csv = df_results_csv.sort_values(['root_cluster', 'leaf_cluster', 'leaf_distance'])
df_results_csv = df_results_csv[['root_cluster', 'leaf_cluster', 'root_distance', 'leaf_distance', 'summary', 'clustering_summary', 'voe_v2_id']]

output_dir = '/content/drive/Shareddrives/00.U-ZERO/03.development/02.コンストラクティブフィードバック/分析V2 Colab/results/with_outlier_detection'
timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
output_file = f'{output_dir}/kmeans_{target}_{num_root_clusters}_{num_leaf_clusters}_{timestamp}.csv'

df_results_csv.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"結果を '{output_file}' に保存しました")

結果を '/content/drive/Shareddrives/00.U-ZERO/03.development/02.コンストラクティブフィードバック/分析V2 Colab/results/with_outlier_detection/kmeans_4-4_8_35_20251018065149.csv' に保存しました
